In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import random
import sys
sys.path.append('..')
from config import *

# Для воспроизводимости
np.random.seed(42)
random.seed(42)

print(f"Конфигурация:")
print(f"  Клиентов: {N_CLIENTS}")
print(f"  Дней: {DAYS}")
print(f"  Доля фрода: {FRAUD_RATIO*100}%")
print(f"  H0 (честные): lognormal(μ={H0_MU}, σ={H0_SIGMA})")
print(f"  H1 (фрод): lognormal(μ={H1_MU}, σ={H1_SIGMA})")

In [ ]:
# Генерация клиентов
clients = pd.DataFrame({
    'client_id': range(1, N_CLIENTS + 1),
    'client_name': [f'Client_{i}' for i in range(1, N_CLIENTS + 1)]
})
print(f"Создано {len(clients)} клиентов")
clients.head()

In [ ]:
# Функция генерации транзакций для одного клиента
def generate_client_transactions(client_id, days, fraud_ratio):
    transactions = []
    
    # Базовое время начала (например, 2024-01-01)
    start_date = datetime(2024, 1, 1)
    
    # Среднее число транзакций в день для этого клиента (от 1 до 5)
    avg_txn_per_day = np.random.uniform(1, 5)
    total_days = days
    
    # Пуассоновский процесс: количество транзакций по дням
    for day in range(total_days):
        n_txns = np.random.poisson(avg_txn_per_day)
        
        for _ in range(n_txns):
            # Определяем, фрод или нет
            is_fraud = np.random.random() < fraud_ratio
            
            # Сумма транзакции в зависимости от типа
            if is_fraud:
                amount = np.random.lognormal(mean=H1_MU, sigma=H1_SIGMA)
            else:
                amount = np.random.lognormal(mean=H0_MU, sigma=H0_SIGMA)
            
            # Время транзакции (случайное время в течение дня)
            txn_time = start_date + timedelta(days=day, seconds=np.random.randint(0, 86400))
            
            transactions.append({
                'client_id': client_id,
                'amount': round(amount, 2),
                'txn_time': txn_time,
                'is_fraud': int(is_fraud)
            })
    
    return transactions

In [ ]:
# Генерация всех транзакций
all_transactions = []

for client_id in range(1, N_CLIENTS + 1):
    if client_id % 20 == 0:
        print(f"Генерация для клиента {client_id} из {N_CLIENTS}")
    txns = generate_client_transactions(client_id, DAYS, FRAUD_RATIO)
    all_transactions.extend(txns)

transactions_df = pd.DataFrame(all_transactions)
print(f"\nВсего транзакций: {len(transactions_df)}")
print(f"Из них фродовых: {transactions_df['is_fraud'].sum()} ({transactions_df['is_fraud'].mean()*100:.2f}%)")
transactions_df.head()

In [ ]:
# Сортировка по времени (важно для потока)
transactions_df = transactions_df.sort_values('txn_time').reset_index(drop=True)

# Добавляем txn_id
transactions_df.insert(0, 'txn_id', range(1, len(transactions_df) + 1))

print("Первые 10 транзакций по времени:")
transactions_df.head(10)

In [ ]:
# Сохранение в CSV
transactions_df.to_csv('../data/transactions_synthetic.csv', index=False)
clients.to_csv('../data/clients.csv', index=False)

print("✅ Данные сохранены:")
print(f"   - data/transactions_synthetic.csv ({len(transactions_df)} строк)")
print(f"   - data/clients.csv ({len(clients)} строк)")

# Статистика
print("\n📊 Статистика по суммам:")
print("Честные транзакции:")
honest = transactions_df[transactions_df['is_fraud'] == 0]['amount']
print(f"  среднее: {honest.mean():.2f}, медиана: {honest.median():.2f}, 95% перцентиль: {honest.quantile(0.95):.2f}")

fraud = transactions_df[transactions_df['is_fraud'] == 1]['amount']
print("Фродовые транзакции:")
print(f"  среднее: {fraud.mean():.2f}, медиана: {fraud.median():.2f}, 95% перцентиль: {fraud.quantile(0.95):.2f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 5))
sns.histplot(honest, bins=50, color='blue', alpha=0.5, label='Честные', log_scale=True)
sns.histplot(fraud, bins=50, color='red', alpha=0.5, label='Фрод', log_scale=True)
plt.xlabel('Сумма транзакции (log scale)')
plt.ylabel('Частота')
plt.title('Распределение сумм транзакций')
plt.legend()
plt.savefig('../reports/distributions_synthetic.png', dpi=150)
plt.show()